# SYNAPSE fp32 fixed-set learnability check

Select an **A100** runtime. Compare shared logits, independent per-pair logits, and the original feature scorer on two fixed A/M pairs at teacher states 100/500. Each starts uniform and receives 50 full-set steps. Coarse mixture scans and virtual/real update checks accompany fitting.

[Protocol and limitations](https://github.com/akashm776/SYNAPSE/blob/main/docs/LEARNABILITY_CHECK.md). This is an overfitting/optimization diagnostic, **not held-out fine-tuning evidence**. No B/D/test evaluation or decoding. Original pilot and audit outputs stay unchanged.

New A100 runtime is unmeasured. The 30-minute budget is per invocation and checked between units, not a hard timeout. Rerun after a budget pause; do not launch the old pilot run/report.


In [ ]:
import os, sys, subprocess, json
from pathlib import Path
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
REPO = Path('/content/SYNAPSE')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/akashm776/SYNAPSE.git', str(REPO)], check=True)
CODE_REF = ''  # On resume, enter the exact revision printed by your FIRST setup.
if CODE_REF:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', CODE_REF], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', 'origin/main'], check=True)
print('Record learnability revision:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.9.1', 'torchvision==0.24.1', 'torchaudio==2.9.1', '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)+'[llm,test]', 'numpy==2.3.5', 'accelerate==1.2.1'], check=True)
subprocess.run([sys.executable, '-c', "import torch; assert torch.cuda.is_available(); print(torch.cuda.get_device_name()); assert 'A100' in torch.cuda.get_device_name()"], check=True)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SOURCE = Path('/content/drive/MyDrive/SynLess/runs/llm_qwen_v2')
OUTPUT = Path('/content/drive/MyDrive/SYNAPSE/diagnostics/qwen_learnability_v1')
CONFIG = REPO / 'configs/learnability_qwen_a100.json'
assert (SOURCE / 'data.json').exists(), 'Use the complete original Drive run, not the public metric archive'
print('Read-only source:', SOURCE)
print('Separate diagnostic output:', OUTPUT)
print(CONFIG.read_text())


## Run or resume

Rerun after a budget pause. Every full-set fitting step is saved. After a runtime restart, restore the recorded CODE_REF/environment first. Do not run two sessions into the same output folder. Full subprocess output is also appended to `console.log` for debugging.


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
command = [sys.executable, '-u', '-m', 'synapse.learnability', '--source', str(SOURCE), '--output', str(OUTPUT), '--config', str(CONFIG)]
with (OUTPUT / 'console.log').open('a') as log:
    with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        returncode = process.wait()
if returncode:
    raise RuntimeError(f'Diagnostic exited {returncode}; inspect the traceback above or {OUTPUT / "console.log"}')
status = json.loads((OUTPUT / 'learnability_manifest.json').read_text())['status']
print('Learnability status:', status)
if status == 'complete':
    print((OUTPUT / 'LEARNABILITY_REPORT.md').read_text())
else:
    print('Rerun this cell to resume. This is not a completed diagnostic yet.')


## Share results

After completion, share `LEARNABILITY_REPORT.md`, `learnability_summary.json`, `learnability_manifest.json`, and `model_buffers.json`. Keep `state_*/fit_*.pt` for resume. The summary includes every scan, per-pair assessment, and step history; weights/checkpoints need not be shared. Do not run the original official-test report again.
